In [1]:
import pandas as pd
df = pd.read_csv('featured.csv', low_memory=False)

In [2]:
df.columns
cols = [
    'event_id', 'league', 'date', 'away_team', 'away_player',
    'away_score', 'home_team', 'home_player', 'home_score', 'total_score',

    'home_avg_score',
    'home_avg_goals_conceded', 'home_win_rate', 'away_avg_score',
    'away_avg_goals_conceded', 'away_win_rate', 'h2h_avg_goals_home',
    'h2h_avg_goals_away', 'h2h_win_rate_home',

    '1_1_home_od', '1_1_draw_od', '1_1_away_od', '1_1_handicap',
    '1_1_add_time', '1_2_home_od', '1_2_draw_od', '1_2_away_od',
    '1_2_handicap', '1_2_add_time', '1_3_over_od', '1_3_under_od',
    '1_3_handicap', '1_3_add_time'
]

In [3]:
df = df[df['league'] == 22614]
df = df[cols]

In [4]:
df = df.dropna(subset=['event_id', 'league', 'date', 'away_team', 'away_player',
    'away_score', 'home_team', 'home_player', 'home_score', 'total_score',

    'home_avg_score',
    'home_avg_goals_conceded', 'home_win_rate', 'away_avg_score',
    'away_avg_goals_conceded', 'away_win_rate', 'h2h_avg_goals_home',
    'h2h_avg_goals_away', 'h2h_win_rate_home',])

Começar o Treinamento do Modelo

Split: Treinamento até 15-abr.



Validação até 30-abr.


Teste dia-a-dia a partir de 1-mai

In [5]:
import optuna
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance

In [6]:
features = [
    'home_avg_score', 'home_avg_goals_conceded', 'home_win_rate',
    'away_avg_score', 'away_avg_goals_conceded', 'away_win_rate',
    'h2h_avg_goals_home', 'h2h_avg_goals_away', 'h2h_win_rate_home'
]

target = ['total_score']

In [7]:
from datetime import datetime

In [8]:
df['date'] = pd.to_datetime(df['date'])
split_val = datetime(2025, 4, 16)
split_test = datetime(2025, 5, 1)

train_df = df[df['date'] <= split_val]
val_df = df[df['date'].between(split_val, split_test, inclusive='neither')]
test_df = df[df['date'] >= split_test]

In [9]:
X_train = train_df[features]
X_val = val_df[features]
X_test = test_df[features]

y_train = train_df[target]
y_val = val_df[target]
y_test = test_df[target]

In [10]:
# # Normalizar os dados (opcional, mas útil para muitos modelos)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_val_scaled = scaler.transform(X_val)

Optuna para tunar params

In [11]:
import optuna
import xgboost as xgb
from optuna.integration import XGBoostPruningCallback
from sklearn.metrics import mean_poisson_deviance

# supondo que X_train_scaled, X_val_scaled, y_train, y_val já estão definidos

def objective(trial):
    # 1. Espaço de busca com nomes reg_* para regularização
    params = {
        'objective': 'count:poisson',
        'eval_metric': 'poisson-nloglik',
        'booster': 'gbtree',
        'seed': 42,  # 2. semente para reprodutibilidade
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 0.9),
        'reg_alpha':  trial.suggest_float('reg_alpha',  0.1, 0.9),
        'max_depth':  trial.suggest_int('max_depth',   3,   5),
        'eta':        trial.suggest_float('eta',       0.01, 0.1),
        'subsample':  trial.suggest_float('subsample', 0.6,  0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
    }

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val,   label=y_val)

    # 3. incluir n_estimators no espaço de busca
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)

    # 4. treinar com pruning e early stopping
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=n_estimators,
        evals=[(dval, 'eval')],
        early_stopping_rounds=50,
        callbacks=[XGBoostPruningCallback(trial, 'eval-poisson-nloglik')],
        verbose_eval=False
    )

    # 5. avaliar com mean_poisson_deviance (minimizar)
    preds = model.predict(dval)
    return mean_poisson_deviance(y_val, preds)


In [12]:
# 6. criar estudo no modo minimize
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, timeout=600)
best_params = study.best_params

[I 2025-05-18 15:31:24,880] A new study created in memory with name: no-name-6a12f613-94be-48f5-953f-42c0cc1683c2
[I 2025-05-18 15:31:25,470] Trial 0 finished with value: 0.9635225534439087 and parameters: {'reg_lambda': 0.7194361054973787, 'reg_alpha': 0.8911070174570944, 'max_depth': 5, 'eta': 0.027309314872962773, 'subsample': 0.6176627662145516, 'colsample_bytree': 0.890598928118929, 'n_estimators': 570}. Best is trial 0 with value: 0.9635225534439087.
[I 2025-05-18 15:31:26,134] Trial 1 finished with value: 0.9634754061698914 and parameters: {'reg_lambda': 0.38456164365746703, 'reg_alpha': 0.6652283244051839, 'max_depth': 5, 'eta': 0.018217525045309824, 'subsample': 0.8161678805672432, 'colsample_bytree': 0.7644930881097451, 'n_estimators': 821}. Best is trial 1 with value: 0.9634754061698914.
[I 2025-05-18 15:31:26,467] Trial 2 finished with value: 0.9657829403877258 and parameters: {'reg_lambda': 0.7892755958286083, 'reg_alpha': 0.18779664054881026, 'max_depth': 5, 'eta': 0.0699

In [13]:
print("Melhores parâmetros:", study.best_params)
print("Melhor mean_poisson_deviance:", study.best_value)

Melhores parâmetros: {'reg_lambda': 0.6477618731150583, 'reg_alpha': 0.6040477652118882, 'max_depth': 3, 'eta': 0.08919947338974499, 'subsample': 0.7141856674076704, 'colsample_bytree': 0.6936740456527389, 'n_estimators': 292}
Melhor mean_poisson_deviance: 0.9627823233604431


In [14]:
# Adicionar parâmetros fixos aos melhores encontrados
final_params = {
    **best_params,
    'objective': 'count:poisson',
    'eval_metric': 'poisson-nloglik'
}

# Converter dados para DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Treinar com early stopping
final_model = xgb.train(
    final_params,
    dtrain,
    num_boost_round=1000,
    evals=[(dval, 'eval')],
    early_stopping_rounds=50,
    verbose_eval=100
)


[0]	eval-poisson-nloglik:2.28576


c:\Users\Enzo\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:31:31] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "n_estimators" } are not used.

  warnings.warn(smsg, UserWarning)


[100]	eval-poisson-nloglik:2.17042
[108]	eval-poisson-nloglik:2.17046


In [15]:
from sklearn.metrics import root_mean_squared_error
import math

from scipy.stats import poisson
import xgboost as xgb

def predict_goal_probabilities(model, input_data, max_goals=5):
    mu = model.predict(xgb.DMatrix(input_data))
    return {
        k: poisson.pmf(k, mu)
        for k in range(max_goals + 1)
    }




# Prever na validação
y_pred = final_model.predict(dval)

# Métricas de avaliação
print("MAE:", mean_absolute_error(y_val, y_pred))
print("RMSE:", root_mean_squared_error(y_val, y_pred))
print("Poisson Deviance:", mean_poisson_deviance(y_val, y_pred))



# Exemplo para uma partida específica
sample_match = X_val[0:1]
probabilities = predict_goal_probabilities(final_model, sample_match)
print("Probabilidades de gols totais:", probabilities)


MAE: 1.7059372663497925
RMSE: 2.141018867492676
Poisson Deviance: 0.9631991386413574
Probabilidades de gols totais: {0: array([0.0085841]), 1: array([0.04084181]), 2: array([0.09715947]), 3: array([0.15408985]), 4: array([0.18328385]), 5: array([0.17440717])}


In [60]:
def goal_handicap(handicap) -> float | None:

    """
    Gets the handicap from the API and solves type errors,
    Also, converts it to a float and returns the current handicap if successful.
    """

    try: 
        if isinstance(handicap, str):
            if ',' in handicap:
                handicap_vals = [float(h.strip()) for h in handicap.split(',')]
                handicap = sum(handicap_vals) / len(handicap_vals)
            
            else:
                handicap = float(handicap.strip())
        
        return handicap
    
    except ValueError as ve:
        print(f"Error converting handicap '{handicap}': {ve}")
        return None


def profit(
    bet_type,
    handicap,
    total_score,
    bet_odd,
    stake: float = 1.0
) -> tuple[float, str]:
    """
    Calcula o lucro (P/L) de uma aposta over/under em linha de gols.
    Retorna (profit_amount, result).
    """

    # 1) Força bet_type a ser string escalar
    bet_type = str(bet_type).strip().lower()
    if bet_type not in ('over', 'under'):
        return 0.0, 'no_bet'

    try:
        # 2) Força numéricos
        handicap    = float(handicap)
        total_score = float(total_score)
        bet_odd     = float(bet_odd)
        stake       = float(stake)

        # 3) Calcula a diferença (outcome)
        if bet_type == 'over':
            diff = total_score - handicap
        else:  # under
            diff = handicap - total_score

        # 4) Decide lucro e resultado
        if diff >= 0.5:
            profit_amount, result = stake * (bet_odd - 1), 'win'
        elif diff == 0.25:
            profit_amount, result = stake * (bet_odd - 1) / 2, 'half_win'
        elif diff == 0:
            profit_amount, result = 0.0, 'push'
        elif diff == -0.25:
            profit_amount, result = - stake / 2, 'half_loss'
        else:  # diff <= -0.5
            profit_amount, result = - stake, 'loss'

        return profit_amount, result

    except Exception as e:
        # mostra só a mensagem do erro
        print(f"Ajuste de resultado inválido: {e}")
        return 0.0, 'error'


def ev(odd, prob):
    try:
        ev = float(odd) * prob -1
    except: 
        ev = None
    return ev

def poisson_goals(
    lambda_pred: float,
    handicap: float
    ) -> tuple[float, float]:

    """
    Calculate the Probability of a given Goal Line (Handicap).
    TODO: Create Poisson Probabilities for other markets.
    """
    def half_goal_handicap(handicap, lambda_pred) -> tuple[float,float]:
        prob_over = 1 - (poisson.cdf(int(handicap), lambda_pred))
        prob_under = (poisson.cdf(int(handicap), lambda_pred))
        
        return float(prob_over), float(prob_under)
    
    def integer_handicap(handicap,lambda_pred) -> tuple[float,float]:
        prob_over_raw = 1 - (poisson.cdf(int(handicap), lambda_pred))
        prob_under_raw = (poisson.cdf(int(handicap) - 1, lambda_pred))
        
        total = prob_over_raw + prob_under_raw
        
        prob_over = prob_over_raw / total
        prob_under = prob_under_raw / total

        return float(prob_over), float(prob_under)
    
    def quarter_handicap(handicap,lambda_pred) -> tuple[float,float]:
        
        lower = handicap - 0.25
        upper = handicap + 0.25
        probs_over = []
        probs_under = []

        for line in lower, upper:
            if line % 1 == 0.5:
                over, under = half_goal_handicap(
                handicap=line,
                lambda_pred=lambda_pred
            )
                
            elif line % 1 == 0.0:  
                over, under = integer_handicap(
                handicap=line,
                lambda_pred=lambda_pred
            )
            
            probs_over.append(over)
            probs_under.append(under)
        
        prob_over = sum(probs_over) / len(probs_over) 
        prob_under = sum(probs_under) / len(probs_under)
        
        return float(prob_over), float(prob_under)

    if handicap % 1 == 0.5:
        prob_over, prob_under = half_goal_handicap(
            handicap=handicap,
            lambda_pred=lambda_pred
        )

    elif handicap % 1 == 0.0:  
        prob_over, prob_under = integer_handicap(
            handicap=handicap,
            lambda_pred=lambda_pred
        )

    elif handicap % 1 in [0.25, 0.75]:  
        prob_over, prob_under = quarter_handicap(
            handicap=handicap,
            lambda_pred=lambda_pred
        )
    
    else:
        print(f'Invalid Handicap used to estimate goal probabilities: {handicap}')
        return None, None
    
    return prob_over, prob_under

In [61]:
threshold_df = val_df.copy()
threshold_df['y_pred'] = y_pred

threshold_df = threshold_df.drop(columns=['1_1_home_od', '1_1_draw_od', '1_1_away_od', '1_1_handicap',
    '1_1_add_time', '1_2_home_od', '1_2_draw_od', '1_2_away_od',
    '1_2_handicap','1_3_add_time'])


In [62]:

prob_over = []
prob_under = []
ev_over = []
ev_under = []

for index, line in threshold_df.iterrows():
    # try:
    over_prob, under_prob = poisson_goals(
        line['y_pred'], goal_handicap(line['1_3_handicap']))
    
    over_ev = ev(odd=line['1_3_over_od'], prob=over_prob)
    under_ev = ev(odd=line['1_3_under_od'], prob=under_prob)

    # except:
    #     over_prob = None
    #     over_ev = None
    #     under_prob = None
    #     under_ev = None

    prob_over.append(over_prob)
    prob_under.append(under_prob)
    ev_over.append(over_ev)
    ev_under.append(under_ev)

threshold_df['prob_over'] = prob_over
threshold_df['prob_under'] = prob_under
threshold_df['ev_over'] = ev_over
threshold_df['ev_under'] = ev_under

Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handicap used to estimate goal probabilities: nan
Invalid Handic

In [63]:
import numpy as np

bets = []
evb = []
betodd = []
for _, row in threshold_df.iterrows():
    try:
        ev_o = float(row['ev_over'])
        ev_u = float(row['ev_under'])
        
        if (ev_o <= 0) and (ev_u <= 0):
            bets.append('no_bet')
            evb.append(0)
            betodd.append(0)
        else:
            
            better = np.where(ev_o > ev_u, 'over', 'under')
            bets.append(better)
            evb.append(np.where(ev_o > ev_u, ev_o, ev_u))
            betodd.append(np.where(ev_o > ev_u, row['1_3_over_od'], row['1_3_under_od']))

    
    except Exception:
        bets.append('no_bet')

threshold_df['bet_type'] = bets
threshold_df['ev_bet'] = evb
threshold_df['bet_odd'] = betodd



In [64]:
full_profit = []
full_outcome = []
for _, row in threshold_df.iterrows():
    line_profit, line_outcome = profit(
        bet_type=row['bet_type'],
        handicap=goal_handicap(row['1_3_handicap']),
        total_score=row['total_score'],
        bet_odd=row['bet_odd']
        )
    full_profit.append(line_profit)
    full_outcome.append(line_outcome)

threshold_df['profit'] = full_profit
threshold_df['outcome'] = full_outcome

Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: could not convert string to float: '-'
Ajuste de resultado inválido: coul

In [65]:
threshold_df.head()

,event_id,league,date,away_team,away_player,away_score,home_team,home_player,home_score,total_score,...,y_pred,prob_over,prob_under,ev_over,ev_under,bet_type,ev_bet,bet_odd,profit,outcome
107773,9823375,22614,2025-04-16 00:04:00,Czechia,Kray,2,USA,Kodak,5,7,...,4.757843,0.516041,0.483959,-0.058225,-0.092577,no_bet,0,0,0.0,no_bet
107774,9823374,22614,2025-04-16 00:04:00,Ukraine,hotShot,3,Ghana,Boulevard,1,4,...,4.231263,0.516507,0.483493,-0.044461,-0.105539,no_bet,0,0,0.0,no_bet
107775,9823376,22614,2025-04-16 00:06:00,Fiorentina,exhausted,0,Napoli,jAke,4,4,...,5.002346,0.513092,0.486908,-0.050780,-0.099220,no_bet,0,0,0.0,no_bet
107776,9823371,22614,2025-04-16 00:06:00,Roma,panch,2,Bologna,GanGsta_Panda,0,2,...,4.638511,0.493969,0.506031,-0.110855,-0.038542,no_bet,0,0,0.0,no_bet
107782,9823437,22614,2025-04-16 00:16:00,USA,Kodak,5,Ukraine,hotShot,0,5,...,4.758032,0.516075,0.483925,-0.058162,-0.092641,no_bet,0,0,0.0,no_bet


In [66]:
threshold_df.to_excel('test.xlsx')

In [70]:
full_train_df = pd.concat([train_df, val_df])

In [76]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import mean_poisson_deviance

# 1. Separe features e target no full train
X_full = full_train_df[features]
y_full = full_train_df[target]

# e no test
X_test = test_df[features]
y_test = test_df[target]


final_model = XGBRegressor(**best_params)

# 3. Treine com .fit()
final_model.fit(X_full, y_full)

# 4. Preveja e avalie no test
y_pred = final_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))
print("Poisson deviance:", mean_poisson_deviance(y_test, y_pred))


MAE: 1.7184659242630005
RMSE: 2.150709629058838
Poisson deviance: 0.9728101491928101
